  1. COCO DATASET
  Images      : 248
  Annotations : 22731
  Categories  : 356  (IDs 0–355)
  ID gaps     : none
  Images with no annotations: 0
  Avg annotations/image: 91.7
  Width  range: 481–5712 px  avg=2799
  Height range: 399–4624 px  avg=2595

  Top 10 most common categories:
    [355]   422  unknown_product
    [ 86]   398  HAVRE KNEKKEBRØD 300G WASA
    [109]   374  KNEKKEBRØD 100 FRØ&HAVSALT 245G WASA
    [100]   368  EVERGOOD CLASSIC FILTERMALT 250G
    [349]   364  KNEKKEBRØD SPORT+ 210G WASA
    [246]   322  HUSMAN KNEKKEBRØD 260G WASA
    [296]   309  FIBER BALANCE 230G WASA
    [271]   307  FRUKOST KNEKKEBRØD 240G WASA
    [132]   300  FRUKOST FULLKORN 320G WASA
    [207]   297  LEKSANDS KNEKKE FIBERBIT 240G

  Bottom 10 least common categories:
    [ 57]     1  MORENEPOTETER GULE 650G BJERTNÆS&HOEL
    [123]     1  ASPARGES GRØNN
    [234]     1  LANO SÅPE 2X125G
    [274]     1  VESTLANDSLEFSA TØRRE 10STK 360G
    [140]     1  SURDEIGKJEKS 100G SÆTRES BESTE
    [317]

# NorgesGruppen Product Detection — Training Pipeline

**H100 NVL Optimized (270 GB VRAM)**

Run each stage independently by clicking the cell. Stages are designed to be run in order (1→6), but you can re-run any individual stage at any time.

| Stage | Description | Time estimate |
|-------|-------------|---------------|
| 1 | Data Exploration | ~10s |
| 2 | Prepare Detection Data | ~1 min |
| 3 | Train Detector (YOLOv8x) | ~2-4 hours |
| 4 | Prepare Classification Data | ~2 min |
| 5 | Train Classifier (EfficientNet-B3) | ~30-60 min |
| 6 | Build Submission Zip | ~10s |
| 7 | Validate Submission | ~5s |

### Submission Constraints (HARD LIMITS)
- **Max zip size (uncompressed):** 420 MB
- **Max weight files:** 3 (`.pt`, `.onnx`, `.safetensors`, `.npy`)
- **Max weight size total:** 420 MB
- **ONNX opset:** ≤ 20 (we use 17)
- **Sandbox:** L4 GPU (24 GB), 4 vCPU, 8 GB RAM, 300s timeout, Python 3.11
- **Scoring:** 70% detection mAP + 30% classification mAP
- **Blocked imports:** os, sys, subprocess, pickle, shutil, yaml, etc.
- **category_id range:** 0–355

---
## Stage 0 — Environment Check
Verify GPU, VRAM, and key packages before starting.

In [ ]:
import torch
import json
from pathlib import Path

print("=" * 60)
print("  ENVIRONMENT CHECK")
print("=" * 60)

# GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  GPU  : {gpu_name}")
    print(f"  VRAM : {vram_gb:.1f} GB")
    if vram_gb >= 90:
        print(f"  → H100-class GPU detected — using FULL optimizations")
    elif vram_gb >= 40:
        print(f"  → Mid-range GPU — will auto-reduce batch sizes")
    else:
        print(f"  → Small GPU — will use conservative settings")
else:
    print("  WARNING: No GPU detected!")

# Packages
print()
try:
    import ultralytics
    print(f"  ultralytics : {ultralytics.__version__}")
except ImportError:
    print("  ultralytics : NOT INSTALLED — run: pip install ultralytics==8.1.0")

try:
    import timm
    print(f"  timm        : {timm.__version__}")
except ImportError:
    print("  timm        : NOT INSTALLED — run: pip install timm==0.9.12")

try:
    import cv2
    print(f"  opencv      : {cv2.__version__}")
except ImportError:
    print("  opencv      : NOT INSTALLED")

try:
    import onnxruntime
    print(f"  onnxruntime : {onnxruntime.__version__}")
except ImportError:
    print("  onnxruntime : NOT INSTALLED (needed for validation only)")

print(f"  torch       : {torch.__version__}")
print(f"  CUDA        : {torch.version.cuda}")

# Data dirs
print()
data_dirs = [
    "data/raw/coco_dataset/train/annotations.json",
    "data/raw/coco_dataset/train/images",
    "data/raw/product_images",
    "data/aug_product_images",
    "data/raw/extra_data/SKU110K_fixed",
]
for d in data_dirs:
    p = Path(d)
    exists = "✓" if p.exists() else "✗"
    print(f"  {exists} {d}")

print("\n" + "=" * 60)

---
## Stage 1 — Data Exploration
Summarize all datasets: COCO annotations, SKU110K, product images, augmented data.

In [ ]:
%run 01_data_exploration.py

---
## Stage 2 — Prepare Detection Data
Convert COCO + SKU110K + aug_product_images → YOLO format.

In [ ]:
%run 02_prepare_detection_data.py

---
## Stage 3 — Train Detector (YOLOv8x)

**H100 NVL optimized settings:**
- Image: 1280×1280 (was 640) — 4× more pixels for small product detection
- Batch: 64 (was 32) — fills ~200+ GB VRAM
- AdamW + cosine LR (was SGD + linear)
- Multi-scale training, MixUp, CopyPaste, RandAugment
- 300 epochs, patience=40

**Edit the settings below before running if needed.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  DETECTOR CONFIG — edit these before running                ║
# ╚══════════════════════════════════════════════════════════════╝

DETECTOR_MODEL  = "xlarge"   # nano / small / medium / large / xlarge
DETECTOR_EPOCHS = 300        # Training epochs
DETECTOR_IMGSZ  = 1280       # Image size (640, 1024, 1280)
DETECTOR_BATCH  = 64         # Batch size (auto-scales for VRAM)
DETECTOR_PATIENCE = 40       # Early stopping patience
DETECTOR_RESUME = False      # Set True to resume from last checkpoint
DETECTOR_NAME   = "nm_yolov8x"  # Run name

In [ ]:
import torch
from pathlib import Path
# Fix: PyTorch 2.7 defaults weights_only=True, ultralytics 8.1.0 needs False
_orig_load = torch.load
def _patched_load(*a, **kw):
    kw.setdefault('weights_only', False)
    return _orig_load(*a, **kw)
torch.load = _patched_load

from ultralytics import YOLO

# ── Auto-scale for available VRAM ──
WEIGHTS_MAP = {
    "nano": "yolov8n.pt", "small": "yolov8s.pt", "medium": "yolov8m.pt",
    "large": "yolov8l.pt", "xlarge": "yolov8x.pt",
}

weights = WEIGHTS_MAP[DETECTOR_MODEL]
imgsz = DETECTOR_IMGSZ
batch = DETECTOR_BATCH

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if vram_gb < 40:
        batch = min(batch, 8)
        imgsz = min(imgsz, 640)
        print(f"  ⚠ VRAM={vram_gb:.0f}GB — reduced to batch={batch}, imgsz={imgsz}")
    elif vram_gb < 90:
        batch = min(batch, 32)
        imgsz = min(imgsz, 1024)
        print(f"  VRAM={vram_gb:.0f}GB — adjusted to batch={batch}, imgsz={imgsz}")
    else:
        print(f"  VRAM={vram_gb:.0f}GB — using full settings: batch={batch}, imgsz={imgsz}")

DATASET_YAML = Path("data/yolo_detection/dataset.yaml")
RUNS_DIR = Path("runs/detect")

assert DATASET_YAML.exists(), f"Dataset not found: {DATASET_YAML} — run Stage 2 first"

# Load model
if DETECTOR_RESUME:
    last_pt = RUNS_DIR / DETECTOR_NAME / "weights" / "last.pt"
    if last_pt.exists():
        print(f"Resuming from {last_pt}")
        model = YOLO(str(last_pt))
    else:
        print(f"last.pt not found, starting fresh with {weights}")
        model = YOLO(weights)
else:
    model = YOLO(weights)

print(f"\nTraining YOLOv8{DETECTOR_MODEL[0].upper()} @ {imgsz}px, batch={batch}, epochs={DETECTOR_EPOCHS}")
print(f"Estimated VRAM: ~{batch * (imgsz/640)**2 * 3:.0f} GB\n")

# ── Train ──
results = model.train(
    data=str(DATASET_YAML),
    epochs=DETECTOR_EPOCHS,
    imgsz=imgsz,
    batch=batch,
    device=0,
    patience=DETECTOR_PATIENCE,
    save=True,
    save_period=25,
    exist_ok=True,
    verbose=True,
    seed=42,

    # Optimizer — AdamW for many-class fine-tuning
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=5,
    warmup_momentum=0.8,
    warmup_bias_lr=0.01,
    cos_lr=True,

    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.2,
    scale=0.9,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,          # No vertical flip for shelf products
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,       # Helps rare classes
    erasing=0.4,
    close_mosaic=20,
    auto_augment="randaugment",

    # Training options
    multi_scale=True,     # Variable resolution — uses extra VRAM
    rect=False,
    half=True,
    label_smoothing=0.1,
    nbs=64,
    cache=False,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    max_det=400,

    # Output
    val=True,
    plots=True,
    project=str(RUNS_DIR),
    name=DETECTOR_NAME,
    workers=8,
)

print("\n" + "=" * 60)
print("  DETECTOR TRAINING COMPLETE")
print("=" * 60)
if results and hasattr(results, 'results_dict'):
    m = results.results_dict
    print(f"  mAP@50   : {m.get('metrics/mAP50', 0):.4f}")
    print(f"  mAP@50-95: {m.get('metrics/mAP50-95', 0):.4f}")
print(f"  Best weights: {RUNS_DIR}/{DETECTOR_NAME}/weights/best.pt")

---
## Stage 3b — Export Detector to ONNX
Export the best YOLOv8 weights to ONNX for submission (opset 17).

In [ ]:
from pathlib import Path
# Fix: PyTorch 2.7 defaults weights_only=True, ultralytics 8.1.0 needs False
_orig_load = torch.load
def _patched_load(*a, **kw):
    kw.setdefault('weights_only', False)
    return _orig_load(*a, **kw)
torch.load = _patched_load

from ultralytics import YOLO

RUNS_DIR = Path("runs/detect")
SUBMISSION = Path("submission")
SUBMISSION.mkdir(exist_ok=True)

best_pt = RUNS_DIR / DETECTOR_NAME / "weights" / "best.pt"
assert best_pt.exists(), f"best.pt not found at {best_pt} — run Stage 3 first"

print(f"Loading {best_pt}...")
model = YOLO(str(best_pt))

print("Exporting to ONNX (opset 17)...")
onnx_path = model.export(
    format="onnx",
    opset=17,
    simplify=True,
    half=True,          # FP16 — smaller file, faster on L4
    dynamic=False,
    imgsz=640,          # Inference at 640 on L4 (24GB) for speed within 300s
)

# Copy to submission/
src = Path(onnx_path)
dst = SUBMISSION / "detector.onnx"
dst.write_bytes(src.read_bytes())

size_mb = dst.stat().st_size / 1024**2
print(f"\nExported: {dst} ({size_mb:.1f} MB)")
if size_mb > 300:
    print(f"  ⚠ Large file! Classifier + detector must total < 420 MB")

# Copy idx_to_category_id.json
idx_map_src = Path("data/yolo_detection/idx_to_category_id.json")
idx_map_dst = SUBMISSION / "idx_to_category_id.json"
if idx_map_src.exists():
    idx_map_dst.write_bytes(idx_map_src.read_bytes())
    print(f"Copied {idx_map_dst}")

---
## Stage 4 — Prepare Classification Data
Crop COCO bounding boxes + product images → classifier train/val splits.

In [ ]:
%run 04_prepare_classification_data.py

---
## Stage 5 — Train Classifier (EfficientNet-B3)

**H100 NVL optimized settings:**
- Model: EfficientNet-B3 at 300×300 (larger than default 224, safe ONNX size ~48 MB)
- Batch: 512 train / 1024 val — fills ~40-60 GB VRAM
- AdamW + cosine LR with 5-epoch warmup
- MixUp + CutMix + RandAugment + RandomErasing
- EMA (exponential moving average)
- Class-weighted sampling for imbalanced classes
- 40 epochs with gradient accumulation (effective batch 1024)

**Using B3 (not B5) to keep ONNX ~48 MB → total submission well under 420 MB.**

**Edit config below before running.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CLASSIFIER CONFIG — edit these before running              ║
# ╚══════════════════════════════════════════════════════════════╝

CLF_MODEL     = "efficientnet_b3"  # efficientnet_b3 (~48MB) or efficientnet_b5 (~120MB)
CLF_IMG_SIZE  = 300                # B3: 300, B5: 456 (B3 native is 300)
CLF_BATCH     = 512                # Train batch (auto-scales for VRAM)
CLF_BATCH_VAL = 1024               # Val batch
CLF_EPOCHS    = 40                 # Training epochs
CLF_LR        = 1e-3               # Learning rate
CLF_WD        = 0.05               # Weight decay
CLF_ACCUM     = 2                  # Gradient accumulation steps
CLF_WARMUP    = 5                  # Warmup epochs
CLF_EMA_DECAY = 0.9998             # EMA decay (0 to disable)
CLF_MIXUP     = 0.2                # MixUp alpha
CLF_CUTMIX    = 1.0                # CutMix alpha

In [ ]:
import json
import math
import time
import copy
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import timm

DATA_DIR = Path("data/classifier")
SUBMISSION = Path("submission")
SUBMISSION.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)

assert (DATA_DIR / "train").exists(), "Run Stage 4 first"

# ── Auto-scale batch for VRAM ──
batch_train = CLF_BATCH
batch_val = CLF_BATCH_VAL
img_size = CLF_IMG_SIZE

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if vram_gb < 40:
        batch_train = min(batch_train, 64)
        batch_val = min(batch_val, 128)
        img_size = min(img_size, 224)
        print(f"  ⚠ VRAM={vram_gb:.0f}GB — reduced settings")
    elif vram_gb < 90:
        batch_train = min(batch_train, 256)
        batch_val = min(batch_val, 512)

print(f"  Model: {CLF_MODEL} @ {img_size}×{img_size}")
print(f"  Batch: {batch_train} train / {batch_val} val (accum={CLF_ACCUM}, eff={batch_train*CLF_ACCUM})")
print(f"  Device: {DEVICE}")

# ── EMA ──
class ModelEMA:
    def __init__(self, model, decay=0.9998):
        self.ema = copy.deepcopy(model)
        self.ema.eval()
        self.decay = decay
        for p in self.ema.parameters():
            p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for ep, mp in zip(self.ema.parameters(), model.parameters()):
            ep.data.mul_(self.decay).add_(mp.data, alpha=1 - self.decay)
    def state_dict(self):
        return self.ema.state_dict()

# ── MixUp / CutMix ──
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[idx], y, y[idx], lam

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    H, W = x.size(2), x.size(3)
    cut_rat = np.sqrt(1. - lam)
    cw, ch = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, y1 = max(0, cx-cw//2), max(0, cy-ch//2)
    x2, y2 = min(W, cx+cw//2), min(H, cy+ch//2)
    x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2-x1)*(y2-y1)/(W*H)
    return x, y, y[idx], lam

def mix_criterion(crit, pred, ya, yb, lam):
    return lam * crit(pred, ya) + (1-lam) * crit(pred, yb)

# ── Transforms ──
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(img_size, scale=(0.4, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.33)),
])

val_tf = transforms.Compose([
    transforms.Resize(int(img_size * 1.15)),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])

train_ds = datasets.ImageFolder(str(DATA_DIR / "train"), transform=train_tf)
val_ds = datasets.ImageFolder(str(DATA_DIR / "val"), transform=val_tf)
n_classes = len(train_ds.classes)
print(f"  Classes: {n_classes}, Train: {len(train_ds)}, Val: {len(val_ds)}")

# ── Weighted sampler ──
class_counts = Counter(train_ds.targets)
weights_per_class = {c: 1.0/math.sqrt(cnt) for c, cnt in class_counts.items()}
sample_weights = [weights_per_class[t] for t in train_ds.targets]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=batch_train, sampler=sampler,
                          num_workers=8, pin_memory=True, drop_last=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=batch_val, shuffle=False,
                        num_workers=8, pin_memory=True, persistent_workers=True)

# ── Model ──
model = timm.create_model(CLF_MODEL, pretrained=True, num_classes=n_classes,
                          drop_rate=0.3, drop_path_rate=0.2)
model = model.to(DEVICE)
params_m = sum(p.numel() for p in model.parameters()) / 1e6
print(f"  Parameters: {params_m:.1f}M")

ema = ModelEMA(model, decay=CLF_EMA_DECAY) if CLF_EMA_DECAY > 0 else None
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=CLF_LR, weight_decay=CLF_WD)

def lr_lambda(epoch):
    if epoch < CLF_WARMUP:
        return (epoch + 1) / CLF_WARMUP
    progress = (epoch - CLF_WARMUP) / max(1, CLF_EPOCHS - CLF_WARMUP)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

# ── Training loop ──
best_val_acc = 0.0
best_ema_acc = 0.0
best_ckpt = Path("runs/classifier_best.pt")
best_ema_ckpt = Path("runs/classifier_best_ema.pt")
best_ckpt.parent.mkdir(exist_ok=True)

t0 = time.time()
print(f"\nTraining {CLF_EPOCHS} epochs...\n")

for epoch in range(1, CLF_EPOCHS + 1):
    model.train()
    rloss, rcorrect, rtotal = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    for bi, (imgs, labels) in enumerate(train_loader):
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        use_mix = np.random.random() < 0.8
        if use_mix:
            if np.random.random() < 0.5:
                imgs, ya, yb, lam = mixup_data(imgs, labels, CLF_MIXUP)
            else:
                imgs, ya, yb, lam = cutmix_data(imgs, labels, CLF_CUTMIX)

        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            out = model(imgs)
            loss = mix_criterion(criterion, out, ya, yb, lam) if use_mix else criterion(out, labels)
            loss = loss / CLF_ACCUM

        scaler.scale(loss).backward()

        if (bi + 1) % CLF_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if ema: ema.update(model)

        rloss += loss.item() * CLF_ACCUM * imgs.size(0)
        preds = out.argmax(dim=1)
        target = ya if use_mix else labels
        rcorrect += (preds == target).sum().item()
        rtotal += imgs.size(0)

    scheduler.step()
    train_loss = rloss / rtotal
    train_acc = rcorrect / rtotal

    # ── Val ──
    results = {}
    for name, eval_model in [("model", model)] + ([("ema", ema.ema)] if ema else []):
        eval_model.eval()
        vc, vt, t5c = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
                    out = eval_model(imgs)
                vc += (out.argmax(1) == labels).sum().item()
                vt += imgs.size(0)
                t5 = out.topk(min(5, n_classes), dim=1).indices
                t5c += (t5 == labels.unsqueeze(1)).any(1).sum().item()
        results[name] = {"top1": vc/vt, "top5": t5c/vt}

    m_acc = results["model"]["top1"]
    e_acc = results.get("ema", {}).get("top1", 0)
    lr_now = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0

    print(f"  Epoch {epoch:3d}/{CLF_EPOCHS}  loss={train_loss:.4f}  "
          f"train={train_acc:.3f}  val={m_acc:.3f}/{e_acc:.3f}(ema)  "
          f"top5={results['model']['top5']:.3f}  lr={lr_now:.2e}  [{elapsed:.0f}s]")

    if m_acc > best_val_acc:
        best_val_acc = m_acc
        torch.save(model.state_dict(), str(best_ckpt))
    if ema and e_acc > best_ema_acc:
        best_ema_acc = e_acc
        torch.save(ema.state_dict(), str(best_ema_ckpt))

print(f"\nBest val: model={best_val_acc:.3f}, ema={best_ema_acc:.3f}")

### Stage 5b — Export Classifier to ONNX

In [ ]:
import json
import torch
import timm
from pathlib import Path
from torchvision import datasets

DATA_DIR = Path("data/classifier")
SUBMISSION = Path("submission")
OUT_ONNX = SUBMISSION / "classifier.onnx"
OUT_MAP = SUBMISSION / "classifier_idx_to_category_id.json"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Determine which weights are better
best_ckpt = Path("runs/classifier_best.pt")
best_ema_ckpt = Path("runs/classifier_best_ema.pt")

use_ema = best_ema_acc >= best_val_acc and best_ema_ckpt.exists()
ckpt_path = best_ema_ckpt if use_ema else best_ckpt
print(f"Using {'EMA' if use_ema else 'model'} weights (acc={max(best_ema_acc, best_val_acc):.3f})")

# Rebuild model and load weights
train_ds_tmp = datasets.ImageFolder(str(DATA_DIR / "train"))
n_classes = len(train_ds_tmp.classes)

model = timm.create_model(CLF_MODEL, pretrained=False, num_classes=n_classes)
model.load_state_dict(torch.load(str(ckpt_path), map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# Export ONNX
dummy = torch.zeros(1, 3, img_size, img_size, device=DEVICE)
torch.onnx.export(
    model, dummy, str(OUT_ONNX),
    opset_version=17,
    input_names=["images"],
    output_names=["logits"],
    dynamic_axes=None,
    do_constant_folding=True,
)

onnx_mb = OUT_ONNX.stat().st_size / 1024**2
print(f"Exported: {OUT_ONNX} ({onnx_mb:.1f} MB)")

# Write category mapping
src_map = DATA_DIR / "classifier_idx_to_category_id.json"
if src_map.exists():
    raw_map = json.loads(src_map.read_text(encoding="utf-8"))
    folder_to_coco = {}
    for folder_name, if_idx in train_ds_tmp.class_to_idx.items():
        coco_cid = raw_map.get(folder_name)
        folder_to_coco[str(if_idx)] = int(coco_cid) if coco_cid is not None else 0
    OUT_MAP.write_text(json.dumps(folder_to_coco, indent=2), encoding="utf-8")
    print(f"Wrote {OUT_MAP}")
else:
    fallback = {str(i): i for i in range(n_classes)}
    OUT_MAP.write_text(json.dumps(fallback, indent=2), encoding="utf-8")
    print(f"WARNING: wrote identity mapping")

# Size check
det_onnx = SUBMISSION / "detector.onnx"
if det_onnx.exists():
    det_mb = det_onnx.stat().st_size / 1024**2
    total = det_mb + onnx_mb
    print(f"\nSize check: detector={det_mb:.1f}MB + classifier={onnx_mb:.1f}MB = {total:.1f}MB")
    if total > 400:
        print(f"  ⚠ CLOSE TO 420 MB LIMIT! Consider FP16 quantization.")
    else:
        print(f"  ✓ Well within 420 MB limit")

---
## Stage 6 — Build Submission Zip
Package everything into `submission_onnx.zip` and validate against all competition rules.

In [ ]:
%run 06_build_submission.py

---
## Stage 7 — Validate Submission
Run the full validator against all competition rules (file limits, banned imports, ONNX opset, etc.).

In [ ]:
%run submission_requirements_test.py --dir submission/

---
## Quick Reference

### Re-run individual stages
Just scroll up and re-run the cell. All stages are independent once their inputs exist.

### Resume interrupted detector training
Set `DETECTOR_RESUME = True` in the Stage 3 config cell, then re-run.

### Adjust for different GPUs
Batch sizes auto-scale based on detected VRAM:
- **>90 GB** (H100): Full settings
- **40-90 GB** (A100): Reduced batch
- **<40 GB** (L4/T4): Conservative settings

### Submission size budget
| File | Typical size |
|------|--------------|
| detector.onnx (YOLOv8x FP16) | ~130 MB |
| classifier.onnx (B3) | ~48 MB |
| run.py + JSONs | <1 MB |
| **Total** | **~180 MB** (limit: 420 MB) |